In [1]:
#Calculator

import ast
import operator

_ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.Mod: operator.mod,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}


def _safe_eval(node):
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("Only numeric constants are allowed")
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("Unsupported expression")


def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        tree = ast.parse(expression, mode="eval")
        result = _safe_eval(tree)
        return str(result)
    except Exception:
        return "Error in calculation"

## Tool 2: Keyword Extractor


In [2]:
#Keyword Extraction

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

In [3]:
#AGENT LOGIC

import logging
import re

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("agent")

_MATH_EXPR_PATTERN = re.compile(r"[0-9.\s+\-*/()%]+")


def _extract_math_expression(query: str) -> str:
    """Pull out the numeric/operator portion of the query, dropping words like 'calculate'."""
    matches = _MATH_EXPR_PATTERN.findall(query)
    return "".join(matches).strip()


def _extract_keyword_text(query: str) -> str:
    """Strip a leading trigger phrase so only the text to analyze remains."""
    lowered = query.lower()
    for phrase in ("extract keywords from", "keywords from", "keywords"):
        idx = lowered.find(phrase)
        if idx != -1:
            return query[idx + len(phrase):].strip()
    return query

In [4]:
# AGENT FUNCTION

def agent(query: str):
    if not isinstance(query, str) or not query.strip():
        logger.warning("Empty or invalid query received")
        return {"type": "error", "result": "Query cannot be empty"}

    query_lower = query.lower()
    logger.info("Received query: %s", query)

    try:
        if "calculate" in query_lower:
            expression = _extract_math_expression(query)
            if not expression:
                return {"type": "error", "result": "No valid expression found to calculate"}
            result = calculator(expression)
            response_type = "error" if result == "Error in calculation" else "calculation"
            logger.info("Routed to calculator, result=%s", result)
            return {"type": response_type, "result": result}

        elif "keywords" in query_lower:
            text = _extract_keyword_text(query)
            result = extract_keywords(text)
            logger.info("Routed to keyword extractor, result=%s", result)
            return {"type": "keywords", "result": result}

        else:
            result = f"No specific tool matched this query. Here it is for reference: '{query}'"
            logger.info("Routed to general response")
            return {"type": "general", "result": result}

    except Exception as e:
        logger.error("Unhandled error while processing query: %s", e)
        return {"type": "error", "result": f"Unexpected error: {e}"}

In [5]:
# Test cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

2026-07-19 21:15:47,068 [INFO] Received query: Calculate 20 + 5
2026-07-19 21:15:47,069 [INFO] Routed to calculator, result=25
2026-07-19 21:15:47,069 [INFO] Received query: Extract keywords from Artificial Intelligence is transforming industries
2026-07-19 21:15:47,070 [INFO] Routed to keyword extractor, result=['intelligence', 'transforming', 'artificial', 'industries']
2026-07-19 21:15:47,071 [INFO] Received query: What is machine learning?
2026-07-19 21:15:47,071 [INFO] Routed to general response


Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['intelligence', 'transforming', 'artificial', 'industries']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': "No specific tool matched this query. Here it is for reference: 'What is machine learning?'"}
--------------------------------------------------


In [6]:
edge_queries = [
    "",
    "Calculate banana",
]

for q in edge_queries:
    print("Query:", repr(q))
    print("Response:", agent(q))
    print("-" * 50)

2026-07-19 21:15:48,001 [WARNING] Empty or invalid query received
2026-07-19 21:15:48,001 [INFO] Received query: Calculate banana


Query: ''
Response: {'type': 'error', 'result': 'Query cannot be empty'}
--------------------------------------------------
Query: 'Calculate banana'
Response: {'type': 'error', 'result': 'No valid expression found to calculate'}
--------------------------------------------------


In [7]:
while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

Enter query (type 'exit' to stop):  calculate 2 and 3


2026-07-19 21:15:59,142 [INFO] Received query: calculate 2 and 3
2026-07-19 21:15:59,142 [INFO] Routed to calculator, result=Error in calculation


Response: {'type': 'error', 'result': 'Error in calculation'}


Enter query (type 'exit' to stop):  calculate 2 % 4


2026-07-19 21:16:13,865 [INFO] Received query: calculate 2 % 4
2026-07-19 21:16:13,866 [INFO] Routed to calculator, result=2


Response: {'type': 'calculation', 'result': '2'}


Enter query (type 'exit' to stop):  exit
